# Семинар 7. Дизайн пилота

Легенда:
продакт ПРАЙМ хочет дать подписчикам Базового тарифа **кэшбэк 5 % на доставку** вместо 1 %.
Кэшбэк оплачивает партнёр доставки, ПРАЙМ ждёт роста оборота в партнёрских сервисах — с него
комиссия.

Продакт: «Давайте включим всем с понедельника и через месяц посмотрим, что выросло».

На лекции мы разобрали, почему так нельзя. Сегодня — дизайн пилота, который можно будет
защитить: гипотеза, метрики, объём, разбиение на группы, проверка процедуры и дизайн-документ.
Итоги этого пилота — на семинаре 8.

| Метка | Что значит |
|---|---|
| **понимать** | спросят на защите и на проверочной |
| **уметь писать** | понадобится в ДЗ-3 и в итоговом проекте |
| **знать, что существует** | достаточно помнить, что такое бывает, и загуглить |

> **Окружение.** Ноутбук работает в окружении `pilot/` с HypEx 1.0.7 — как его поставить,
> написано в README темы 07. Из корня `prime-monitor`: `uv run --project pilot jupyter lab`.

---

## 1. Данные

Кандидаты в пилот — 12 000 владельцев действующих подписок Базового тарифа на 31.08.2026.
Для каждого есть история за два четырёхнедельных периода **до** пилота:

| Колонка | Что это |
|---|---|
| `spend_prev2`, `spend_prev` | оборот в партнёрских сервисах, ₽: 06.07–02.08 и 03.08–30.08 |
| `purchases_prev2`, `purchases_prev` | число покупок за те же периоды |
| `sessions_prev2`, `sessions_prev` | сессии в сервисах подписки |
| `members` | с кем поделились подпиской: 0, 1 или 2 |
| `acquisition_channel` | канал привлечения, **бывает пусто** — как в `prime.clients` |

Данные синтетические и лежат в схеме `prime` учебного сервера. Описание — в `данные/пилот/README.md`.
Первая ячейка подключает HypEx и базу, вторая заводит загрузку таблиц.

In [ ]:
import sys
from pathlib import Path

# Окружение pilot/ не устанавливает пакет prime: у него pandas 3, а HypEx 1.0.7
# работает только с pandas 2.2. Поэтому src/ подключаем к пути сами —
# ищем папку src/prime вверх от текущей.
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "prime").is_dir()), None)
if root is None:
    raise RuntimeError("Не нашёл src/prime. Запустите JupyterLab из корня prime-monitor:\n"
                       "uv run --project pilot jupyter lab")
sys.path.insert(0, str(root / "src"))

import hypex

if hypex.__version__ != "1.0.7":
    raise RuntimeError(f"Нужен HypEx 1.0.7, а запущен {hypex.__version__} из {sys.executable}.\n"
                       "Закройте JupyterLab и запустите из корня prime-monitor: uv run --project pilot jupyter lab")

import warnings

import numpy as np
import pandas as pd
from sqlalchemy import text

from prime.config import get_engine

warnings.filterwarnings("ignore")          # HypEx шумит предупреждениями pandas
pd.set_option("display.width", 160)
engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


def F(obj):
    """Результаты HypEx — объекты Dataset. Для pandas-операций берём .data."""
    return obj.data if hasattr(obj, "data") else obj


print("HypEx", hypex.__version__, "· pandas", pd.__version__, "· подключились как:",
      q("select current_user as login").iloc[0, 0])

In [ ]:
# Таблицы пилота лежат в схеме prime. Если сервер не отвечает — поменяйте
# False на True: те же таблицы скачаются из репозитория курса.
SERVER_IS_DOWN = False

RAW = ("https://raw.githubusercontent.com/tikhomirovd/python-for-ba-hse-2026/master/"
       "%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D0%B5/%D0%BF%D0%B8%D0%BB%D0%BE%D1%82/")


def load(name: str) -> pd.DataFrame:
    """Таблица пилота целиком; client_id — строкой, как в любой выгрузке."""
    df = pd.read_csv(RAW + f"{name}.csv.gz") if SERVER_IS_DOWN else q(f"select * from prime.{name}")
    if "client_id" in df:
        df["client_id"] = df["client_id"].astype(str)
    return df

In [ ]:
cand = load("pilot_candidates")

# Одна строка — один клиент? Проверяем до любых расчётов: Л4.
assert cand["client_id"].is_unique, "в кандидатах повторяются клиенты"

print(len(cand), "кандидатов")
cand.head()

In [ ]:
# Какая доля кандидатов в августе не купила ничего? Ноль — это значение,
# а не пропуск: метрика определена для всех.
zero_share = (cand["spend_prev"] == ЗАПОЛНИТЕ).mean()

print(f"оборот 0 в августе: {zero_share:.1%}")
print("без канала привлечения:", cand["acquisition_channel"].isna().sum())

**6,1 % ничего не купили, у 611 нет канала.** Оба факта понадобятся: первый — чтобы не потерять
нули в метрике, второй — при стратификации.

---

## 2. Гипотеза и метрики

Слайды 9–10 и 16–17 лекции. Заполните по шаблону — своими словами, дважды щёлкнув по ячейке.

✍️ **Гипотеза.** Если …, для …, то … изменится не меньше чем на … за …, потому что … .

✍️ **Основная метрика** (одна, с единицами и окном): …

✍️ **Вспомогательные:** …

✍️ **Стражи** (что не должно ухудшиться — хотя бы два): …

✍️ **Решающее правило.** Раскатываем, если … . Не раскатываем, если … .

**Понимать.** Основная метрика одна, и по ней считается всё остальное: объём, MDE, решение.
Метрика «оборот среди открывших push» не годится: кто открыл push, решило само воздействие —
в контроле таких людей нет.

> **Зачем это в работе.** Решающее правило, записанное до запуска, экономит неделю споров после:
> заказчик заранее согласился, какой результат что означает.

---

## 3. Объём выборки

Формула со слайда 24:

$$n \approx 2\,(z_{1-\alpha/(2k)} + z_{1-\beta})^2\,\frac{\sigma^2}{\text{MDE}^2}$$

где $k$ — число сравнений с контролем: две тестовые группы — два сравнения, и на каждое
по Бонферрони приходится $\alpha/2$. Квантиль нормального распределения — `NormalDist().inv_cdf`
из стандартной библиотеки, без scipy.

In [ ]:
import math
from statistics import NormalDist

z = NormalDist().inv_cdf


def n_per_group(sigma: float, mde: float, alpha: float = 0.05, power: float = 0.8,
                comparisons: int = 1) -> int:
    """Клиентов на КАЖДУЮ группу, чтобы заметить эффект mde с заданной мощностью."""
    za = z(1 - alpha / (2 * comparisons))
    # Второй квантиль отвечает за мощность. Какой у него аргумент?
    zb = z(ЗАПОЛНИТЕ)
    return math.ceil(2 * (za + zb) ** 2 * sigma ** 2 / mde ** 2)


sigma = cand["spend_prev"].std()
print(f"среднее {cand['spend_prev'].mean():.0f} ₽, sigma {sigma:.0f} ₽")

# Учебник: sigma = 1, mde = 0.5 при одном сравнении — около 63 на группу.
print(n_per_group(1, 0.5))

In [ ]:
# Сколько нужно при разных MDE: одно сравнение и два.
grid = pd.DataFrame({"mde": [100, 150, 200, 250, 300]})
grid["одно сравнение"] = grid["mde"].map(lambda m: n_per_group(sigma, m))
grid["два сравнения"] = grid["mde"].map(lambda m: n_per_group(sigma, m, comparisons=ЗАПОЛНИТЕ))
grid["есть на группу"] = len(cand) // 3
grid

**MDE 200 ₽ — это 6,3 % от среднего, и на него не хватает:** на два сравнения нужно 5 138
на группу, а есть 4 000.

Вариантов три: больше клиентов, меньше групп, больший MDE. Есть четвёртый — уменьшить
$\sigma$. Дисперсия падает в $1-\rho^2$ раз, где $\rho$ — корреляция метрики с её прошлым
значением (слайд 25). Но сентябрьского оборота ещё нет — поэтому корреляцию берём между
двумя **прошлыми** периодами.

In [ ]:
# Корреляция июля с августом — оценка того, как август будет связан с сентябрём.
rho = cand["spend_prev2"].corr(cand[ЗАПОЛНИТЕ])
sigma_reduced = sigma * math.sqrt(1 - rho ** 2)

print(f"rho = {rho:.3f}, дисперсия ниже на {rho**2:.1%}")
print("нужно на группу, два сравнения, со снижением:",
      n_per_group(sigma_reduced, 200, comparisons=2))

**3 306 при 4 000 — проходит.** Пилот выполним только со снижением дисперсии, и это решение
дизайна: CUPED по августовскому обороту записываем в дизайн-документ до запуска.

**Уметь писать.** `n_per_group` — четыре строки, которые отвечают на вопрос «хватит ли нам
клиентов» до того, как кто-то потратил месяц.

> **Зачем это в работе.** Чаще всего выборка задана — «у нас 12 тысяч клиентов на тарифе».
> Тогда тот же расчёт в обратную сторону даёт MDE: эффект меньше этого пилот не увидит,
> и об этом надо сказать до запуска, а не после.

---

## 4. Разбиение на группы в HypEx

Слайды 21–22. Три группы по трети, стратификация по каналу привлечения.

Пропуск в канале — отдельная страта «без канала». Если его не заполнить, 611 клиентов выпадут
из стратификации: та же история, что с `groupby` на третьем семинаре.

In [ ]:
from hypex import AATest
from hypex.dataset import Dataset, InfoRole, StratificationRole, TargetRole

# Чем заполняем пропуск? Договорённость курса — отдельная строка «без канала».
cand["acquisition_channel"] = cand["acquisition_channel"].fillna(ЗАПОЛНИТЕ)

ds = Dataset(
    roles={
        "client_id": InfoRole(str),
        "acquisition_channel": StratificationRole(),
        "spend_prev": TargetRole(),
        "sessions_prev": TargetRole(),
    },
    data=cand,
    default_role=InfoRole(),
)

# Доли групп — списком, в сумме единица. Первая станет контролем. ~25 секунд.
aa = AATest(groups_sizes=ЗАПОЛНИТЕ, stratification=True, n_iterations=30).execute(ds)
F(aa.resume)

In [ ]:
split = F(aa.best_split)
shares = pd.crosstab(split["acquisition_channel"], split["split"], normalize="columns")

# Стратификация обещает одинаковые доли каналов в группах. Проверяем это обещание.
gap = (shares.max(axis=1) - shares.min(axis=1)).max()
assert gap < 0.002, f"доли каналов разошлись на {gap:.4f}"

print(split["split"].value_counts().to_string())
shares.round(4)

Колонки `aa test` — про процедуру по всем 30 делениям, `best split` — про одно отобранное
деление. Читаем обе.

**Понимать.** 30 итераций — это оценка доли ложных отвержений с погрешностью 0,04 (приложение к лекции, слайд 35).
Отличить честную процедуру от процедуры с ошибкой 9 % по ним нельзя. Перед настоящим запуском —
больше итераций или своя симуляция, как в следующем разделе.

### Ловушка: словарь ролей

`Dataset` дописывает роли по умолчанию **прямо в переданный словарь**. Если собрать словарь
один раз и передать его второму датасету с другими колонками, получите `RoleColumnError`.

In [ ]:
roles = {"client_id": InfoRole(str), "spend_prev": TargetRole()}
Dataset(roles=roles, data=cand, default_role=InfoRole())

print(len(roles), "ключей в словаре после первого Dataset")
try:
    Dataset(roles=roles, data=cand[["client_id", "spend_prev"]], default_role=InfoRole())
except Exception as e:
    print(type(e).__name__, "— словарь уже знает про все колонки cand")

# Как правильно: передавать копию.
Dataset(roles=dict(ЗАПОЛНИТЕ),
        data=cand[["client_id", "spend_prev"]], default_role=InfoRole())
print("с копией — работает")

---

## 5. Проверка процедуры руками: обе ошибки

Слайд 20. `AATest` проверяет ошибку I рода. Ошибку II рода в HypEx не проверить — внесение
эффекта пишем сами. Заодно увидим, что снижение дисперсии делает с мощностью.

Делим кандидатов случайно на две группы по 4 000 много раз и на каждом делении считаем
t-критерий по августовскому обороту:

* без эффекта — доля отвержений должна быть около 5 %;
* с эффектом +200 ₽ в одной группе — около заявленной мощности 80 %.

In [ ]:
from scipy import stats


def rejection_share(y: np.ndarray, n_each: int, effect: float, k: int = 1000,
                    seed: int = 7) -> float:
    """Доля делений, на которых t-критерий отверг H0 при alpha = 0.05."""
    rng = np.random.default_rng(seed)
    hits = 0
    for _ in range(k):
        perm = rng.permutation(len(y))
        a, b = y[perm[:n_each]], y[perm[n_each:2 * n_each]]
        # Эффект вносим в первую группу. Как?
        p = stats.ttest_ind(ЗАПОЛНИТЕ, b, equal_var=False).pvalue
        hits += p < 0.05
    return hits / k


y = cand["spend_prev"].to_numpy()
print(f"ошибка I рода:  {rejection_share(y, 4000, effect=0):.1%}")
print(f"мощность, +200: {rejection_share(y, 4000, effect=200):.1%}")

**5,6 % и 77 %.** Процедура честная: 5,6 % от пяти по тысяче делений не отличить.
А мощность ниже заявленных 80 % — 4 000 на группу для MDE 200 ₽ мало, как и обещала формула.

Теперь то же с CUPED. Ковариата — июльский оборот, $\theta = \mathrm{cov}(Y, X)/\mathrm{var}(X)$.

In [ ]:
x = cand["spend_prev2"].to_numpy()
theta = np.cov(y, x)[0, 1] / x.var(ddof=1)

# Вычитаем отклонение ковариаты от СРЕДНЕГО, а не саму ковариату.
y_cuped = y - theta * (x - ЗАПОЛНИТЕ)

print(f"theta = {theta:.3f}, дисперсия ниже на {1 - y_cuped.var() / y.var():.1%}")
print(f"ошибка I рода:  {rejection_share(y_cuped, 4000, effect=0):.1%}")
print(f"мощность, +200: {rejection_share(y_cuped, 4000, effect=200):.1%}")

**Мощность выросла с 77 до 91 %, ошибка I рода осталась у пяти процентов — 4,8 %.** Ради этого
CUPED и записывают в дизайн.

**Уметь писать.** `rejection_share` — десять строк, которые проверяют любую процедуру: другой
критерий, другую метрику, другую единицу. Подставьте свою — и узнаете, честна ли она, до запуска.

> **Зачем это в работе.** Аналитики редко ошибаются в формуле t-критерия и часто — в том,
> к чему его применяют. Симуляция на своих же исторических данных ловит обе ошибки сразу.

---

## 6. Размер выборки в HypEx

Приложение к лекции, слайд 36. `MinSampleSize` — калькулятор: `.calc()`, не `.execute()`. MDE — в рублях.

In [ ]:
from hypex.dataset import TreatmentRole
from hypex.executor.calculators import MinSampleSize

pair = split[split["split"] != "test_2"]

# Без колонки групп калькулятор не знает, откуда брать дисперсии.
try:
    MinSampleSize(mde=200.0, alpha=0.05, equal_variance=True).calc(
        data=Dataset(roles={"client_id": InfoRole(str), "spend_prev": TargetRole()},
                     data=pair, default_role=InfoRole()))
except Exception as e:
    print(type(e).__name__, ":", e)

ds_pair = Dataset(
    roles={"client_id": InfoRole(str), "split": ЗАПОЛНИТЕ, "spend_prev": TargetRole()},
    data=pair, default_role=InfoRole(),
)
MinSampleSize(mde=200.0, alpha=0.05, equal_variance=True).calc(data=ds_pair)

**4 296 против 4 242 у нашей функции на одно сравнение.** Калькулятор оценивает дисперсию
внутри групп разбиения, наша функция — по всей выборке. На трёх группах сразу HypEx считает
поправку своим способом, и число с формулой не сравнить — поэтому сверяем на паре.

**Знать, что существует.** `equal_variance=False` — по умолчанию — ищет размер выборки
симуляцией при разных дисперсиях групп. Медленнее, но честнее, когда группы неоднородны.

---

## 7. Закреплённые клиенты

Партнёр «Мобильный оператор» попросил не менять условия 300 своим абонентам из списка:
у них свой договор. Они обязаны остаться в контроле, остальные делятся вокруг них.
Такая колонка размечается `ConstGroupRole`: у закреплённых — метка группы, у остальных — пропуск.

In [ ]:
fixed_ids = cand.loc[cand["acquisition_channel"] == "partner_telecom", "client_id"].head(300)
mask = cand["client_id"].isin(fixed_ids)

# Так делать НЕЛЬЗЯ: numpy приведёт np.nan к строке "nan", пропусков не останется.
wrong = np.where(mask, "control", np.nan)
print("пропусков в неправильной колонке:", pd.isna(wrong).sum())

In [ ]:
from hypex.dataset import ConstGroupRole

fixed = pd.Series([None] * len(cand), dtype=object)
fixed[mask.to_numpy()] = "control"
cand["fixed_group"] = fixed
print("пропусков:", cand["fixed_group"].isna().sum(), "· закреплённых:", cand["fixed_group"].notna().sum())

ds_fixed = Dataset(
    roles={"client_id": InfoRole(str), "fixed_group": ЗАПОЛНИТЕ,
           "spend_prev": TargetRole()},
    data=cand, default_role=InfoRole(),
)
aa_fixed = AATest(groups_sizes=[1/3, 1/3, 1/3], n_iterations=10).execute(ds_fixed)
sp = F(aa_fixed.best_split)
pd.crosstab(sp["fixed_group"].fillna("свободные"), sp["split"])

Все 300 — в контроле, остальные разошлись вокруг. С колонкой из `np.where` разбиение падает
с `IndexError: single positional indexer is out-of-bounds` — по сообщению причину не угадать.

---

## 8. Дизайн-документ

Слайд 30. Заполните таблицу числами из этого ноутбука. Это то, что фиксируется **до** запуска;
на семинаре 8 мы будем сверять итоги именно с ней.

| Пункт | Наш пилот |
|---|---|
| гипотеза | … |
| единица рандомизации | … |
| группы, доли, страты | … |
| основная метрика | … |
| стражи | … |
| α, мощность, MDE | … |
| поправка на множественность | … |
| оценщик | … |
| окно | … |
| решающее правило | … |

---

## 9. Компонент: `src/prime/pilot.py`

Расчёт объёма переезжает в ваш репозиторий. Создайте файл `src/prime/pilot.py` и перенесите
туда `n_per_group`. Импорты сверху — только стандартная библиотека, поэтому модуль работает
и в основном окружении `prime-monitor`, и в `pilot/`:

```python
import math
from statistics import NormalDist

z = NormalDist().inv_cdf
```

Допишите туда же обратную функцию — `mde_for_n(sigma, n, alpha=0.05, power=0.8, comparisons=1)`:
какой минимальный эффект заметит пилот с $n$ клиентами на группу. Это ★1 ниже.

После этого в любом ноутбуке:

```python
from prime.pilot import n_per_group, mde_for_n
```

Коммит — в `main`:

```
git add src/prime/pilot.py && git commit -m "расчёт объёма пилота с семинара 7"
```

На семинаре 8 в этот файл добавятся интервал, SRM и CUPED.

---

## Что унести с сегодняшнего дня

1. **Гипотеза — с числом и сроком.** «Оборот +200 ₽ за 4 недели», а не «повысит лояльность».
2. **Объём считается до запуска.** 4 000 на группу при MDE 200 ₽ не хватает; со снижением
   дисперсии на 36 % — хватает.
3. **Снижение дисперсии оценивают по двум прошлым периодам:** текущего ещё нет.
4. **Стратификация выравнивает доли по построению,** а пропуск в страте — отдельная страта.
5. **Процедуру проверяют по обеим ошибкам.** Ошибку II рода HypEx не проверяет — симуляция
   с внесённым эффектом, десять строк.
6. **Две ловушки HypEx:** колонка для `ConstGroupRole` без `np.where`, словарь ролей — копией.

---

## Хвост: ваш вариант ДЗ-3

Дальше — работа с открытым концом: сколько успеете на паре, остальное дома. Ничего из этого
не сдаётся.

У каждого свой пилот для ДЗ-3: номер варианта — в `prime.hw3_variants`, данные — в `prime.hw3_pilot`.
Там есть и итоги, но **сегодня только дизайн**: колонки `*_prev` и `*_prev2`. Итоги — после
семинара 8.

In [ ]:
variant = int(q("select variant from prime.hw3_variants where login = current_user").iloc[0, 0])
offer = q("select * from prime.hw3_offers where variant = :v", v=variant)
mine = q("select * from prime.hw3_pilot where variant = :v", v=variant)

print("вариант", variant, "—", offer["offer"].iloc[0])
print(len(mine), "клиентов, группы:", mine["grp"].value_counts().to_dict())

In [ ]:
# Задача 1. MDE вашего пилота при 80 % мощности: одно сравнение, без снижения
# дисперсии и с CUPED по прошлому периоду. Достаточно ли его, чтобы заметить +3 % к обороту?
my_mde = None   # ← ваш расчёт

print(my_mde)

In [ ]:
# Задача 2. Проверьте процедуру своего пилота функцией rejection_share:
# ошибку I рода и мощность для эффекта, равного вашему MDE с CUPED.

### Задача 3. Дизайн-документ своего пилота

Заполните таблицу из раздела 8 для своего варианта. Она войдёт в ДЗ-3 первым разделом —
и будет честной, только если написана до того, как вы посмотрели на итоги.

---

## Задачи со звёздочкой

Пять задач на то, что на лекции было на экране, а руками вы этого не делали.
**Не сдаются и не оцениваются.**

Считаются на общих данных `cand` — они у всех одинаковые. Проверка знает ответ, но не
показывает его: замените `None` своим расчётом и запустите ячейку.

In [ ]:
# Это просто запустите
ANSWERS = {
    "★1": "80280e4e77",
    "★2": "b1585fdb27",
    "★3": "5ee92f21d3",
    "★4": "4523540f15",
    "★5": "4f33b49459"
}

In [ ]:
# Это тоже
import hashlib

import numpy as np
import pandas as pd


def _norm(x):
    if isinstance(x, bool):
        return repr(x)
    if isinstance(x, float):
        return f"{x:.2f}"
    if isinstance(x, int):
        return str(x)
    if isinstance(x, dict):
        return "{" + ",".join(f"{_norm(k)}:{_norm(v)}" for k, v in sorted(x.items(), key=lambda kv: _norm(kv[0]))) + "}"
    if isinstance(x, (list, tuple)):
        return "[" + ",".join(_norm(v) for v in x) + "]"
    if isinstance(x, np.integer):
        return str(int(x))
    if isinstance(x, np.floating):
        return f"{float(x):.2f}"
    if isinstance(x, np.bool_):
        return repr(bool(x))
    return repr(x)


def check(task: str, got) -> None:
    """Сверить ответ с эталоном. Эталоны лежат хешами — подсмотреть нельзя."""
    if got is None:
        print(f"{task}: не решено")
        return
    digest = hashlib.sha256(_norm(got).encode()).hexdigest()[:10]
    if not ANSWERS:
        print(f"HASH:{task}={digest}")
    elif digest == ANSWERS.get(task):
        print(f"{task}: верно")
    else:
        print(f"{task}: не сходится, получилось {got!r}")


print("проверка готова")

### ★ 1. Обратная задача: какой MDE даёт выборка

Напишите `mde_for_n(sigma, n, alpha=0.05, power=0.8, comparisons=1)` — она же пойдёт
в `src/prime/pilot.py`. Какой MDE у нашего пилота при 4 000 на группу, двух сравнениях
и без снижения дисперсии?

**Ответ:** рубли с двумя знаками — `round(x, 2)`.

In [ ]:
answer = None   # ← ваш расчёт

check("★1", answer)

### ★ 2. Сколько итераций АА-теста нужно

Приложение к лекции, слайд 35. Стандартная ошибка оценки доли отвержений — $\sqrt{\alpha(1-\alpha)/k}$.
Какое наименьшее $k$ даёт погрешность не больше 0,01 при $\alpha = 0{,}05$?

**Ответ:** целое число.

In [ ]:
answer = None   # ← ваш расчёт

check("★2", answer)

### ★ 3. Мощность по формуле

Симуляция в разделе 5 дала около 78 %. Посчитайте мощность по формуле: при эффекте MDE
и $n$ на группу статистика критерия в среднем равна $\text{MDE}/(\sigma\sqrt{2/n})$, мощность —
$\Phi\bigl(\text{MDE}/(\sigma\sqrt{2/n}) - z_{1-\alpha/2}\bigr)$. MDE = 200, $n$ = 4 000, одно
сравнение, $\sigma$ — по августу.

**Ответ:** доля с двумя знаками — `round(x, 2)`.

In [ ]:
answer = None   # ← ваш расчёт

check("★3", answer)

### ★ 4. Сколько недель набирать

Представьте, что клиентов не 12 000 сразу, а приходит 600 новых в неделю, и брать в пилот можно
только новых. Сколько **целых** недель набора нужно, чтобы в каждой из трёх групп оказалось
3 306 клиентов — объём со снижением дисперсии из раздела 3?

**Ответ:** целое число.

In [ ]:
answer = None   # ← ваш расчёт

check("★4", answer)

### ★ 5. Ошибка I рода на два сравнения

Каждое из двух сравнений с контролем проводится на уровне 0,05. Какова вероятность хотя бы
одного ложного отвержения, если эффекта нет нигде? Считайте сравнения независимыми.
А на уровне 0,025 каждое — как в нашем дизайне?

**Ответ:** список из двух долей с двумя знаками — `[round(a, 2), round(b, 2)]`.

In [ ]:
answer = None   # ← ваш расчёт

check("★5", answer)